<a href="https://colab.research.google.com/github/SeYoon0118/pathmnist-aug/blob/main/plots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PathMNIST 그래프 코드 (11주차, 이세윤)
이 노트북은 **위에서부터 한 셀씩 차례로** 실행한다.

| 셀 | 하는 일 |
|---|---|
| 1 | 드라이브 연결, 저장소 받기, 한글 글꼴 |
| 2 | 경로 설정|
| 3 | 공통 색·저장 함수 |
| 4 | 로그 CSV 불러오기 |
| 5 | ① 학습 곡선 |
| 6 | 요약표 (최고 epoch, 과적합 차이, 시간) |
| 7 | ② A/B/C 비교 막대그래프 + Δ(조건 − A) |
| 8 | 체크포인트로 예측 만들기 (기본: **검증셋**) |
| 9 | ③ 혼동 행렬 |
| 10 | ④ 클래스별 F1 |
| 11 | (선택) 가짜 B/C로 3조건 레이아웃 시험 |

테스트셋은 설정 확정 후 조건별 한 번만 평가하기로 했으므로, 이 노트북은 기본으로 **검증셋**만 쓴다.

In [ ]:
# [셀 1] 드라이브 연결 + 저장소 + 한글 글꼴 (Colab에서 실행)
from google.colab import drive
drive.mount('/content/drive')

!pip install -q medmnist
import os
if not os.path.exists('/content/pathmnist-aug'):
    !git clone -q https://github.com/SeYoon0118/pathmnist-aug.git /content/pathmnist-aug
%cd /content/pathmnist-aug
!git pull -q

# 한글 글꼴(나눔고딕): 그래프 제목·축 이름이 □□로 깨지지 않게
!apt-get -qq install -y fonts-nanum > /dev/null

In [ ]:
# [셀 2] 경로 설정 — BASE를 실제 공유 폴더 위치로 바꿔주세요
# (공유받은 폴더라면 드라이브에서 '바로가기를 내 드라이브에 추가'를 해야 이 경로로 보여요)
BASE =  "/content/drive/MyDrive/PathMNIST_결과"

RESULTS_DIR = os.path.join(BASE, "results")      # log_*.csv
CKPT_DIR    = os.path.join(BASE, "checkpoints")  # best_*.pt
FIG_DIR     = os.path.join(BASE, "figures")      # 그래프 저장 위치
os.makedirs(FIG_DIR, exist_ok=True)

def find_file(name):
    """results/ 와 checkpoints/ 중 파일이 있는 곳을 찾아 경로를 돌려준다."""
    for d in (RESULTS_DIR, CKPT_DIR, BASE):
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name}을(를) {RESULTS_DIR}, {CKPT_DIR}에서 찾지 못했어요. BASE 경로를 확인하세요.")

print("log :", find_file("log_A_seed0.csv"))
print("ckpt:", find_file("best_A_seed0.pt"))

In [ ]:
# [셀 3] 공통 설정: 조건별 고정 색, 글꼴, 그림 저장 함수
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import LinearSegmentedColormap

_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(_font):
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams.update({
    "axes.unicode_minus": False,          # 한글 글꼴에서 마이너스(−) 깨짐 방지
    "figure.facecolor": "white", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold", "font.size": 10,
    "lines.linewidth": 2,
})

EXPS = ["A", "B", "C"]
# 조건마다 색을 고정 (어느 그래프에서든 A=파랑, B=주황, C=청록) — 색약에도 구분되는 조합
COLOR = {"A": "#2a78d6", "B": "#eb6834", "C": "#1baf7a", "D": "#6250d6"}
LABEL = {"A": "A 원본", "B": "B 방향", "C": "C 방향+색감", "D": "D 색감만"}
CM_CMAP = LinearSegmentedColormap.from_list("blue_seq", ["#fcfcfb", "#86b6ef", "#2a78d6", "#0d366b"])

# 클래스 이름: 저장소 evaluation.py와 같은 순서(0~8)
from evaluation import CLASS_NAMES, class_report
CLASS_KO = ["지방", "배경", "잔해", "림프구", "점액", "평활근", "정상 점막", "암 연관 간질", "선암 상피"]

DEMO = False   # 셀 11(가짜 B/C)을 실행하면 True가 되고, 그동안은 그림을 저장하지 않음

def save_fig(fig, name):
    """figures/ 에 날짜를 붙여 PNG로 저장 (예: curve_seed0_1112.png)"""
    if DEMO:
        print("(DEMO 모드라 저장 안 함)")
        return
    path = os.path.join(FIG_DIR, f"{name}_{datetime.date.today():%m%d}.png")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("저장:", path)

In [ ]:
# [셀 4] 로그 CSV 불러오기 — 있는 파일만 읽음 (지금은 A seed0 하나)
LOG_COLS = ["epoch", "train_time_sec", "train_loss_running", "train_eval_loss", "train_eval_acc",
            "train_eval_f1", "val_loss", "val_acc", "val_f1", "eval_time_sec"]

def load_logs(exps=EXPS, seeds=(0, 1, 2)):
    """{(조건, 시드): DataFrame} 형태로 반환. 없는 파일은 건너뜀."""
    logs = {}
    for e in exps:
        for s in seeds:
            try:
                df = pd.read_csv(find_file(f"log_{e}_seed{s}.csv"))
            except FileNotFoundError:
                continue
            missing = set(LOG_COLS) - set(df.columns)
            assert not missing, f"log_{e}_seed{s}.csv에 열이 없음: {missing}"
            logs[(e, s)] = df
    print("불러온 로그:", sorted(logs))
    return logs

logs = load_logs()
logs[("A", 0)].tail(3)

In [ ]:
# [셀 5] ① 학습 곡선: Macro F1 / Loss, 실선=검증, 점선=원본 학습 1만 장, ★=검증 F1 최고 epoch
def plot_learning_curves(logs, seed=0, exps=EXPS):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    panels = [("f1", "Macro F1", "train_eval_f1", "val_f1"),
              ("loss", "Loss (Cross-Entropy)", "train_eval_loss", "val_loss")]
    for ax, (key, title, tr_col, va_col) in zip(axes, panels):
        for e in exps:
            if (e, seed) not in logs:
                continue
            df = logs[(e, seed)]
            c = COLOR[e]
            ax.plot(df["epoch"], df[va_col], color=c, label=f"{LABEL[e]} · 검증")
            ax.plot(df["epoch"], df[tr_col], color=c, ls="--", lw=1.5, alpha=0.8, label=f"{LABEL[e]} · 학습 1만")
            b = df["val_f1"].idxmax()          # BestCheckpoint와 같은 기준(검증 F1 최고, 동점이면 먼저)
            ax.plot(df.loc[b, "epoch"], df.loc[b, va_col], marker="*", ms=14, color=c,
                    markeredgecolor="white", zorder=5)
        ax.set_title(title)
        ax.set_xlabel("epoch")
    axes[0].legend(fontsize=8, loc="lower right")
    fig.suptitle(f"학습 곡선 (seed {seed})  —  ★ 검증 Macro F1 최고 epoch", y=1.02)
    fig.tight_layout()
    return fig

fig = plot_learning_curves(logs, seed=0)
save_fig(fig, "curve_seed0")
plt.show()

In [ ]:
# [셀 6] 요약표: 로그에서 '최고 검증 F1 epoch' 기준 값 + 시간
def summarize_logs(logs):
    rows = []
    for (e, s), df in sorted(logs.items()):
        b = df["val_f1"].idxmax()
        r = df.loc[b]
        rows.append({
            "exp": e, "seed": s, "best_epoch": int(r["epoch"]),
            "val_acc": r["val_acc"] * 100, "val_f1": r["val_f1"] * 100,         # %
            "train_eval_f1": r["train_eval_f1"] * 100,
            "gap_f1": (r["train_eval_f1"] - r["val_f1"]) * 100,               # 과적합 정도(%p)
            "total_train_min": df["train_time_sec"].sum() / 60,
            "epoch_time_sec": df["train_time_sec"].mean(),
            "eval_time_sec": df["eval_time_sec"].mean(),
        })
    return pd.DataFrame(rows)

def aggregate(summary):
    """조건별 평균·표준편차 (시드가 1개면 std는 NaN)"""
    return summary.drop(columns="seed").groupby("exp").agg(["mean", "std"]).round(3)

summary = summarize_logs(logs)
display(summary.round(3))
display(aggregate(summary))

In [ ]:
# [셀 7] ② A/B/C 비교 막대그래프
# 그림 1: 절댓값(0부터 시작)  /  그림 2: A 대비 차이 Δ — 1%p 안팎 차이는 그림 2에서 보임
METRICS = [("val_f1", "검증 Macro F1 (%)"), ("val_acc", "검증 Accuracy (%)"), ("epoch_time_sec", "epoch당 학습 시간 (초)")]

def plot_compare_bars(summary, metrics=METRICS, exps=EXPS, title="A/B/C 비교"):
    present = [e for e in exps if e in set(summary["exp"])]
    g = summary.groupby("exp")
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.2 * len(metrics), 4))
    for ax, (col, name) in zip(np.atleast_1d(axes), metrics):
        m = g[col].mean().reindex(present)
        sd = g[col].std().reindex(present).fillna(0)       # 시드 1개면 오차막대 없음
        n = g[col].count().reindex(present)
        x = np.arange(len(present))
        ax.bar(x, m, width=0.6, color=[COLOR[e] for e in present],
               yerr=sd, capsize=4, error_kw={"ecolor": "#52514e", "lw": 1})
        for xi, (mv, sv) in enumerate(zip(m, sd)):
            txt = f"{mv:.2f}" + (f" ± {sv:.2f}" if sv > 0 else "")
            ax.annotate(txt, (xi, mv + sv), xytext=(0, 4), textcoords="offset points",
                        ha="center", va="bottom", fontsize=9, color="#0b0b0b")
        ax.set_xticks(x, [f"{LABEL[e]}\n(n={n[e]})" for e in present])
        ax.set_title(name)
        ax.set_ylim(0, (m + sd).max() * 1.18)
        ax.grid(axis="x", visible=False)
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    return fig

def delta_table(summary):
    """Δ = 조건 − A (%p), 색감 추가 효과 = C − B, 시간 증가율(%) = (조건 − A) ÷ A × 100"""
    m = summary.groupby("exp")[["val_f1", "val_acc", "epoch_time_sec"]].mean()
    out = {}
    pairs = [("B", "A"), ("C", "A"), ("C", "B")]
    for hi, lo in pairs:
        if hi in m.index and lo in m.index:
            out[f"{hi}−{lo}"] = {
                "ΔMacro F1 (%p)": m.loc[hi, "val_f1"] - m.loc[lo, "val_f1"],
                "ΔAccuracy (%p)": m.loc[hi, "val_acc"] - m.loc[lo, "val_acc"],
                "학습 시간 증가율 (%)": (m.loc[hi, "epoch_time_sec"] - m.loc[lo, "epoch_time_sec"]) / m.loc[lo, "epoch_time_sec"] * 100,
            }
    return pd.DataFrame(out).T.round(2)

def plot_delta(dt):
    if dt.empty:
        print("비교할 조건이 2개 이상 있어야 Δ 그래프를 그릴 수 있어요.")
        return None
    fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.6))
    for ax, col in zip(axes, dt.columns):
        v = dt[col]
        y = np.arange(len(v))
        # 좋아짐/나빠짐을 색으로만 말하지 않도록 값 숫자를 함께 표시
        if "시간" in col:   # 시간은 '비용'이라 좋고 나쁨 색 대신 회색
            colors = ["#898781"] * len(v)
        else:              # 성능: 좋아짐 파랑 / 나빠짐 빨강
            colors = ["#2a78d6" if t >= 0 else "#e34948" for t in v]
        ax.barh(y, v, height=0.55, color=colors)
        for yi, t in zip(y, v):
            ax.text(t, yi, f" {t:+.2f} ", va="center", ha="left" if t >= 0 else "right", fontsize=9)
        ax.axvline(0, color="#898781", lw=1)
        ax.set_yticks(y, v.index)
        ax.invert_yaxis()
        ax.set_title(col)
        lim = max(abs(v).max() * 1.6, 0.5)
        ax.set_xlim(-lim, lim)
        ax.grid(axis="y", visible=False)
    fig.suptitle("A 대비 차이 (색감 추가 효과 = C−B)", y=1.03)
    fig.tight_layout()
    return fig

fig = plot_compare_bars(summary, title="A/B/C 비교 — 검증셋, 최고 검증 F1 epoch 기준")
save_fig(fig, "compare_bars_val")
plt.show()

dt = delta_table(summary)
display(dt)
fig = plot_delta(dt)
if fig is not None:
    save_fig(fig, "delta_val")
    plt.show()

In [ ]:
# [셀 8] 체크포인트로 예측 만들기 (혼동 행렬·클래스별 F1용)
# 기본은 검증셋. 테스트셋은 설정 확정 후 팀이 정한 시점에만 ALLOW_TEST=True로!
import torch
from torch.utils.data import DataLoader
from medmnist import PathMNIST
from pathmnist_transforms import get_transform
from model import PathMNISTCNN
from evaluation import evaluate, class_report

ALLOW_TEST = False
device = "cuda" if torch.cuda.is_available() else "cpu"
_loaders = {}

def get_loader(split):
    if split == "test" and not ALLOW_TEST:
        raise RuntimeError("테스트셋은 최종 평가 때만! ALLOW_TEST=False 상태예요.")
    if split not in _loaders:
        ds = PathMNIST(split=split, download=True, transform=get_transform("A"))   # 평가용은 항상 원본
        _loaders[split] = DataLoader(ds, batch_size=256, shuffle=False, num_workers=2)
    return _loaders[split]

def get_preds(exp, seed, split="val"):
    """best_{exp}_seed{seed}.pt로 예측. 한 번 만든 결과는 results/에 npz로 저장해 재사용."""
    cache = os.path.join(RESULTS_DIR, f"preds_{exp}_seed{seed}_{split}.npz")
    if os.path.exists(cache):
        z = np.load(cache)
        return {"y_true": z["y_true"], "y_pred": z["y_pred"]}
    ckpt = torch.load(find_file(f"best_{exp}_seed{seed}.pt"), map_location=device)
    model = PathMNISTCNN().to(device)
    model.load_state_dict(ckpt["model_state"])
    res = evaluate(model, get_loader(split), device)
    print(f"{exp} seed{seed} [{split}] epoch {ckpt['epoch']} | acc {res['acc']:.4f} | "
          f"macro F1 {res['macro_f1']:.4f} (체크포인트 기록값 {ckpt['val_macro_f1']:.4f})")
    np.savez(cache, y_true=res["y_true"], y_pred=res["y_pred"])
    return res

preds = {}
for e in EXPS:
    try:
        preds[(e, 0)] = get_preds(e, 0, "val")
    except FileNotFoundError:
        pass
print("예측 있는 조건:", sorted(preds))

In [ ]:
# [셀 9] ③ 혼동 행렬: 행(정답) 기준 비율(%) — 대각선 = 클래스별 Recall
def plot_confusion(preds, seed=0, exps=EXPS, split="val"):
    present = [e for e in exps if (e, seed) in preds]
    fig, axes = plt.subplots(1, len(present), figsize=(6.2 * len(present), 5.6), squeeze=False)
    ticks = [f"{i} {n}" for i, n in enumerate(CLASS_KO)]
    for ax, e in zip(axes[0], present):
        _, cm = class_report(preds[(e, seed)]["y_true"], preds[(e, seed)]["y_pred"])
        pct = cm / cm.sum(axis=1, keepdims=True) * 100
        ax.imshow(pct, cmap=CM_CMAP, vmin=0, vmax=100)
        for i in range(9):
            for j in range(9):
                if pct[i, j] >= 0.5 or i == j:     # 0.5% 미만 칸은 숫자 생략(가독성)
                    ax.text(j, i, f"{pct[i, j]:.0f}" if pct[i, j] >= 1 else f"{pct[i, j]:.1f}",
                            ha="center", va="center", fontsize=8,
                            color="white" if pct[i, j] > 55 else "#0b0b0b")
        ax.set_xticks(range(9), range(9))
        ax.set_yticks(range(9), ticks if ax is axes[0][0] else range(9))
        ax.set_xlabel("예측 클래스")
        if ax is axes[0][0]:
            ax.set_ylabel("정답 클래스")
        ax.set_title(f"{LABEL[e]}  (Acc {np.mean(preds[(e, seed)]['y_true'] == preds[(e, seed)]['y_pred'])*100:.1f}%)")
        ax.grid(False)
    split_ko = {"val": "검증셋", "test": "테스트셋"}[split]
    fig.suptitle(f"혼동 행렬 — {split_ko}, seed {seed}  (각 행 합 = 100%)", y=1.0)
    fig.tight_layout()
    return fig

fig = plot_confusion(preds, seed=0)
save_fig(fig, "confusion_val_seed0")
plt.show()

In [ ]:
# [셀 10] ④ 클래스별 F1 (+ 조건이 2개 이상이면 A 대비 Δ)
def class_f1_table(preds, seed=0, exps=EXPS):
    cols = {}
    for e in exps:
        if (e, seed) in preds:
            rows, _ = class_report(preds[(e, seed)]["y_true"], preds[(e, seed)]["y_pred"])
            cols[e] = [r["f1"] * 100 for r in rows]
    return pd.DataFrame(cols, index=[f"{i} {n}" for i, n in enumerate(CLASS_KO)])

def plot_class_f1(tbl, seed=0, split="val"):
    present = list(tbl.columns)
    has_delta = "A" in present and len(present) > 1
    fig, axes = plt.subplots(1, 2 if has_delta else 1, figsize=(14 if has_delta else 9, 4.6),
                             squeeze=False, gridspec_kw={"width_ratios": [3, 2]} if has_delta else None)
    ax = axes[0][0]
    w = 0.8 / len(present)
    x = np.arange(len(tbl))
    for k, e in enumerate(present):
        ax.bar(x + (k - (len(present) - 1) / 2) * w, tbl[e], width=w * 0.9, color=COLOR[e], label=LABEL[e])
    ax.set_xticks(x, tbl.index, rotation=35, ha="right")
    ax.set_ylim(0, 118)
    ax.set_yticks(range(0, 101, 20))
    ax.set_ylabel("F1 (%)")
    ax.set_title("클래스별 F1")
    ax.grid(axis="x", visible=False)
    if len(present) > 1:
        ax.legend(fontsize=8, loc="upper center", ncol=len(present), frameon=False)
    else:
        for xi, v in zip(x, tbl[present[0]]):
            ax.text(xi, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)
    if has_delta:
        ax2 = axes[0][1]
        others = [e for e in present if e != "A"]
        h = 0.8 / len(others)
        y = np.arange(len(tbl))
        for k, e in enumerate(others):
            d = tbl[e] - tbl["A"]
            ax2.barh(y + (k - (len(others) - 1) / 2) * h, d, height=h * 0.9, color=COLOR[e], label=f"{e}−A")
        ax2.axvline(0, color="#898781", lw=1)
        ax2.set_yticks(y, tbl.index)
        ax2.invert_yaxis()
        ax2.set_xlabel("ΔF1 (%p)  ← 나빠짐 | 좋아짐 →")
        ax2.set_title("A 대비 클래스별 F1 변화")
        ax2.legend(fontsize=8)
        ax2.grid(axis="y", visible=False)
    split_ko = {"val": "검증셋", "test": "테스트셋"}[split]
    fig.suptitle(f"클래스별 F1 — {split_ko}, seed {seed}", y=1.02)
    fig.tight_layout()
    return fig

tbl = class_f1_table(preds, seed=0)
display(tbl.round(2))
fig = plot_class_f1(tbl, seed=0)
save_fig(fig, "class_f1_val_seed0")
plt.show()

## (선택) 셀 11 — 가짜 B/C로 3조건 레이아웃 시험
실제 B/C 결과가 나오기 전에 **그래프 모양만** 확인하는 용도예요. A 로그·예측을 살짝 흔들어 만든 **가짜 데이터**이고, 이 셀을 실행하면 `DEMO=True`가 되어 그림이 저장되지 않아요.
끝나면 **셀 3부터 다시 실행**해서 DEMO를 끄고 실제 데이터로 돌아오세요.

In [ ]:
# [셀 11] 가짜 B/C (해석 금지! 모양 확인용)
DEMO = True
rng = np.random.default_rng(0)
demo_logs, demo_preds = {}, {}
for s in (0, 1, 2):                               # 시드 3개처럼 만들어 오차막대도 확인
    for e, shift, slow in [("A", 0, 1.0), ("B", 0.006, 1.15), ("C", 0.01, 1.35)]:
        df = logs[("A", 0)].copy()
        noise = rng.normal(0, 0.004, len(df))
        for col in ["val_f1", "val_acc"]:
            df[col] = (df[col] + shift + noise).clip(0, 1)
        df["train_eval_f1"] = (df["train_eval_f1"] - shift * 2).clip(0, 1)
        df["train_time_sec"] = df["train_time_sec"] * slow * rng.uniform(0.97, 1.03)
        demo_logs[(e, s)] = df
yt, yp = preds[("A", 0)]["y_true"], preds[("A", 0)]["y_pred"]
for e, flip in [("A", 0.0), ("B", 0.02), ("C", 0.03)]:
    p = yp.copy()
    m = rng.random(len(p)) < flip
    p[m] = yt[m]                                   # 일부 오답을 정답으로 바꿔 '좋아진 척'
    demo_preds[(e, 0)] = {"y_true": yt, "y_pred": p}

plot_learning_curves(demo_logs, seed=0); plt.show()
ds = summarize_logs(demo_logs)
plot_compare_bars(ds, title="[DEMO] A/B/C 비교"); plt.show()
display(delta_table(ds)); plot_delta(delta_table(ds)); plt.show()
plot_confusion(demo_preds, seed=0); plt.show()
plot_class_f1(class_f1_table(demo_preds, seed=0), seed=0); plt.show()